# ECG CNN Classifier — Training on PTB-XL

**Goal:** Train a 1D CNN to classify 12-lead ECG records into 5 diagnostic superclasses:
- Normal (NORM)
- Myocardial Infarction (MI)
- Conduction Disturbance (CD)
- Hypertrophy (HYP)
- ST-T Abnormality (STTC)

**Dataset:** [PTB-XL](https://physionet.org/content/ptb-xl/) — 21,837 records from 18,885 patients, 12-lead, 10 seconds each.

**Kaggle setup:**
1. Go to **Settings → Accelerator → GPU T4 x2** (free tier)
2. Add the PTB-XL dataset (see Cell 2 for options)
3. Run all cells

## 1. Install dependencies

In [ ]:
!pip install -q wfdb scipy scikit-learn

## 2. Get PTB-XL data

**Option A — Add as Kaggle dataset (recommended):**
1. Download PTB-XL from https://physionet.org/content/ptb-xl/1.0.3/
2. Zip it and upload to Kaggle Datasets
3. Click **+ Add Input** in this notebook → select your dataset
4. Update `DATA_DIR` below to point to the unzipped folder

**Option B — Direct download from PhysioNet** (slow, ~5 GB):

In [ ]:
# ── Configure paths ──────────────────────────────────────────
import os
from pathlib import Path

# Update this to wherever your PTB-XL data lives on Kaggle.
# If you added it as a Kaggle dataset, it'll be under /kaggle/input/<dataset-name>/
DATA_DIR = Path("/kaggle/input/ptbxl")

# If the data isn't found there, fall back to downloading
if not DATA_DIR.exists():
    print("Dataset not found at", DATA_DIR)
    print("Attempting direct download from PhysioNet...")
    !wget -q -P /kaggle/working/ https://physionet.org/files/ptb-xl/1.0.3/ptb-xl-a-large-publicly-available-ecg-dataset-1.0.3.zip
    !unzip -q /kaggle/working/ptb-xl-a-large-publicly-available-ecg-dataset-1.0.3.zip -d /kaggle/working/
    DATA_DIR = Path("/kaggle/working/ptb-xl-a-large-publicly-available-ecg-dataset-1.0.3")

print(f"Data directory: {DATA_DIR}")
print("Contents:", os.listdir(DATA_DIR)[:20])

## 3. Preprocessing pipeline

**SHARED MODULE** — this exact same preprocessing is used for both training and inference.
Changing anything here silently changes model behavior at inference time.

Steps: bandpass filter (0.5–45 Hz) → resample to 500 Hz → segment into 10s windows → z-score normalize per lead.

In [ ]:
import numpy as np
from scipy.signal import butter, filtfilt, resample
from typing import Tuple

# ── Constants ─────────────────────────────────────────────
TARGET_FS = 500            # Hz
BANDPASS_LOW = 0.5         # Hz
BANDPASS_HIGH = 45.0       # Hz
SEGMENT_LENGTH_SEC = 10
NUM_LEADS = 12


def bandpass_filter(signal: np.ndarray, fs: int,
                    low=BANDPASS_LOW, high=BANDPASS_HIGH, order=4) -> np.ndarray:
    nyq = 0.5 * fs
    b, a = butter(order, [low / nyq, high / nyq], btype="band")
    return np.apply_along_axis(lambda x: filtfilt(b, a, x), axis=0, arr=signal)


def resample_signal(signal: np.ndarray, fs_orig: int, fs_target: int) -> np.ndarray:
    n_target = int(len(signal) * fs_target / fs_orig)
    return resample(signal, n_target, axis=0)


def segment_signal(signal: np.ndarray, fs: int, seg_sec=SEGMENT_LENGTH_SEC) -> np.ndarray:
    seg_samples = fs * seg_sec
    n = len(signal)
    n_segs = n // seg_samples
    if n_segs == 0:
        padded = np.zeros((seg_samples, signal.shape[1]))
        padded[:n] = signal
        return padded.reshape(1, seg_samples, signal.shape[1])
    trimmed = signal[: n_segs * seg_samples]
    return trimmed.reshape(n_segs, seg_samples, signal.shape[1])


def normalize_segments(segments: np.ndarray) -> np.ndarray:
    mean = segments.mean(axis=1, keepdims=True)
    std = segments.std(axis=1, keepdims=True)
    std = np.where(std == 0, 1.0, std)
    return (segments - mean) / std


def preprocess_ecg(signal: np.ndarray, fs: int) -> np.ndarray:
    """Full pipeline: filter → resample → segment → normalize."""
    if signal.ndim == 1:
        signal = signal.reshape(-1, 1)
    filtered = bandpass_filter(signal, fs)
    if fs != TARGET_FS:
        filtered = resample_signal(filtered, fs, TARGET_FS)
    segments = segment_signal(filtered, TARGET_FS)
    segments = normalize_segments(segments)
    return segments  # (num_segments, 5000, num_leads)


print("Preprocessing pipeline defined.")

## 4. Load PTB-XL metadata and build label mapping

PTB-XL ships `ptbxl_database.csv` with per-record diagnostic labels. We map each record to one of the 5 diagnostic superclasses. Records with multiple labels get the first matching superclass (prioritizing MI > CD > HYP > STTC > NORM).

In [ ]:
import pandas as pd
import ast

# ── Label mapping ───────────────────────────────────────────
LABEL_MAP = {
    "NORM": 0,
    "MI":   1,
    "CD":   2,
    "HYP":  3,
    "STTC": 4,
}
LABEL_NAMES = {v: k for k, v in LABEL_MAP.items()}


def load_ptbxl_metadata(data_dir: Path):
    """Load metadata using scp_statements.csv (the authoritative mapping shipped with PTB-XL)."""
    # 1. Load the SCP→superclass mapping from scp_statements.csv
    scp_statements = pd.read_csv(data_dir / "scp_statements.csv", index_col=0)
    diagnostic_codes = scp_statements[scp_statements.diagnostic == 1]
    # Build: SCP code string → superclass string  (e.g. 'AMI' → 'MI')
    code_to_superclass = dict(zip(diagnostic_codes.index, diagnostic_codes.diagnostic_class))

    # 2. Load record metadata
    df = pd.read_csv(data_dir / "ptbxl_database.csv", index_col="ecg_id")
    df["scp_codes"] = df["scp_codes"].apply(ast.literal_eval)

    # 3. Assign each record a single superclass label (priority: MI > CD > HYP > STTC > NORM)
    record_ids = []
    labels = {}
    file_paths = {}
    skipped = 0

    for ecg_id, row in df.iterrows():
        scp_codes = row["scp_codes"]
        assigned = None
        for supercls in ["MI", "CD", "HYP", "STTC", "NORM"]:
            if any(code_to_superclass.get(code) == supercls for code in scp_codes.keys()):
                assigned = supercls
                break

        if assigned is None:
            skipped += 1
            continue

        rid = str(ecg_id).zfill(5)
        record_ids.append(rid)
        labels[rid] = LABEL_MAP[assigned]
        file_paths[rid] = row["filename_hr"]  # 500 Hz path from the CSV

    print(f"Loaded {len(record_ids)} records ({skipped} skipped)")
    print(f"SCP codes mapped via scp_statements.csv: {len(code_to_superclass)} diagnostic codes")

    # Class distribution
    from collections import Counter
    dist = Counter(labels.values())
    for cls_id in sorted(dist):
        print(f"  {LABEL_NAMES[cls_id]:6s}: {dist[cls_id]:5d}  ({dist[cls_id]/len(labels)*100:.1f}%)")

    return record_ids, labels, file_paths


record_ids, labels, file_paths = load_ptbxl_metadata(DATA_DIR)


## 5. PyTorch Dataset

In [ ]:
import torch
from torch.utils.data import Dataset
import wfdb
from typing import List, Dict


class PTBXLDataset(Dataset):
    """Loads WFDB records using filename_hr from the CSV, applies shared preprocessing."""

    def __init__(self, data_dir: Path, record_ids: List[str],
                 labels: Dict[str, int], file_paths: Dict[str, str]):
        self.data_dir = data_dir
        self.record_ids = record_ids
        self.labels = labels
        self.file_paths = file_paths  # ecg_id → 'records500/00000/00001_hr' from CSV
        self._cache = {}

    def __len__(self):
        return len(self.record_ids)

    def __getitem__(self, idx):
        rid = self.record_ids[idx]
        if rid in self._cache:
            signal, label = self._cache[rid]
        else:
            # filename_hr from ptbxl_database.csv gives the exact path
            # e.g. 'records500/00000/00001_hr' — append to data_dir
            full_path = str(self.data_dir / self.file_paths[rid])
            sig, fs = wfdb.rdsamp(full_path)  # (samples, leads), int
            preprocessed = preprocess_ecg(sig, fs)
            signal = preprocessed[0]          # first segment: (5000, 12)
            label = self.labels[rid]
            self._cache[rid] = (signal, label)

        tensor = torch.from_numpy(signal.T).float()  # (12, 5000) for Conv1d
        return tensor, label


print(f"Dataset class defined. {len(record_ids)} records available.")


## 6. CNN model architecture

In [ ]:
import torch.nn as nn


class ECGClassifier(nn.Module):
    """
    1D CNN: 4 conv+pool blocks → global avg pool → dense head.
    Input:  (batch, 12, 5000)
    Output: (batch, num_classes)
    """

    def __init__(self, num_leads=12, num_classes=5, segment_samples=5000):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(num_leads, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.squeeze(-1)
        return self.classifier(x)


# Quick sanity check
_test = ECGClassifier()
_out = _test(torch.randn(2, 12, 5000))
print(f"Model OK — output shape: {_out.shape}  |  Parameters: {sum(p.numel() for p in _test.parameters()):,}")
del _test, _out

## 7. Training and evaluation functions

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, n = 0.0, 0
    for signals, targets in loader:
        signals, targets = signals.to(device), targets.to(device)
        optimizer.zero_grad()
        loss = criterion(model(signals), targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n += 1
    return total_loss / max(n, 1)


@torch.no_grad()
def evaluate(model, loader, device, num_classes):
    model.eval()
    all_preds, all_labels = [], []
    for signals, targets in loader:
        signals = signals.to(device)
        preds = model(signals).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(targets.numpy())

    all_preds, all_labels = np.array(all_preds), np.array(all_labels)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    cm = confusion_matrix(all_labels, all_preds, labels=list(range(num_classes)))
    fn_rates, fp_rates = [], []
    for i in range(num_classes):
        tp = cm[i, i]
        fn = cm[i].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        fn_rates.append(fn / (tp + fn) if (tp + fn) > 0 else 0.0)
        fp_rates.append(fp / (tn + fp) if (tn + fp) > 0 else 0.0)

    return {
        "accuracy": acc, "f1_macro": f1,
        "fn_rate": np.mean(fn_rates), "fp_rate": np.mean(fp_rates),
        "per_class_fn": fn_rates, "per_class_fp": fp_rates,
    }


print("Training & evaluation functions defined.")

## 8. Train the model

**Hyperparameters** — adjust as needed. Start conservative; iterate based on results.

In [ ]:
import time
import csv

# ── Hyperparameters ──────────────────────────────────────────
EPOCHS       = 25
BATCH_SIZE   = 32
LR           = 1e-3
VAL_SPLIT    = 0.2

# ── Device ───────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── Dataset & loaders ────────────────────────────────────────
dataset = PTBXLDataset(DATA_DIR, record_ids, labels, file_paths)
val_size = int(len(dataset) * VAL_SPLIT)
train_size = len(dataset) - val_size
train_set, val_set = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = torch.utils.data.DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {train_size}  |  Val: {val_size}  |  Batch size: {BATCH_SIZE}")

# ── Model, loss, optimizer ───────────────────────────────────
num_classes = len(LABEL_MAP)
model = ECGClassifier(num_leads=12, num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

# ── Logging setup ────────────────────────────────────────────
log_path = "/kaggle/working/training_log.csv"
log_fields = ["epoch", "train_loss", "val_acc", "val_f1", "val_fn", "val_fp", "lr", "epoch_time_s"]
with open(log_path, "w", newline="") as f:
    csv.writer(f).writerow(log_fields)

# ── Training loop ────────────────────────────────────────────
best_f1 = 0.0
history = {"train_loss": [], "val_acc": [], "val_f1": [], "val_fn": [], "val_fp": []}
prev_lr = optimizer.param_groups[0]["lr"]

for epoch in range(EPOCHS):
    t0 = time.time()

    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_m = evaluate(model, val_loader, device, num_classes)
    scheduler.step(val_m["f1_macro"])

    epoch_time = time.time() - t0
    current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss)
    history["val_acc"].append(val_m["accuracy"])
    history["val_f1"].append(val_m["f1_macro"])
    history["val_fn"].append(val_m["fn_rate"])
    history["val_fp"].append(val_m["fp_rate"])

    saved = ""
    if val_m["f1_macro"] > best_f1:
        best_f1 = val_m["f1_macro"]
        torch.save(model.state_dict(), "/kaggle/working/best_model.pth")
        saved = " \u2605"

    lr_note = ""
    if current_lr != prev_lr:
        lr_note = f" | LR reduced: {prev_lr:.2e} -> {current_lr:.2e}"
    prev_lr = current_lr

    print(
        f"Epoch {epoch+1:3d}/{EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"Acc: {val_m['accuracy']:.4f} | "
        f"F1: {val_m['f1_macro']:.4f} | "
        f"FN: {val_m['fn_rate']:.4f} | "
        f"FP: {val_m['fp_rate']:.4f} | "
        f"Time: {epoch_time:.1f}s{saved}{lr_note}"
    )

    with open(log_path, "a", newline="") as f:
        csv.writer(f).writerow([
            epoch + 1, train_loss, val_m["accuracy"], val_m["f1_macro"],
            val_m["fn_rate"], val_m["fp_rate"], current_lr, round(epoch_time, 2)
        ])

print(f"\nDone. Best F1: {best_f1:.4f}")
print(f"Full training log saved to {log_path}")


## 9. Plot training curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

epochs_range = range(1, EPOCHS + 1)

# Loss
axes[0].plot(epochs_range, history["train_loss"], label="Train Loss")
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

# Accuracy & F1
axes[1].plot(epochs_range, history["val_acc"], label="Accuracy")
axes[1].plot(epochs_range, history["val_f1"], label="F1 (macro)")
axes[1].set_title("Validation Metrics")
axes[1].set_xlabel("Epoch")
axes[1].legend()

# FN / FP rates
axes[2].plot(epochs_range, history["val_fn"], label="FN rate", color="red")
axes[2].plot(epochs_range, history["val_fp"], label="FP rate", color="orange")
axes[2].set_title("False-Negative / False-Positive Rates")
axes[2].set_xlabel("Epoch")
axes[2].legend()

plt.tight_layout()
plt.savefig("/kaggle/working/training_curves.png", dpi=150)
plt.show()
print("Saved: training_curves.png")

## 10. Per-class breakdown

In [ ]:
final_metrics = evaluate(model, val_loader, device, num_classes)

print("Per-class metrics:")
print(f"{'Class':<8s}  {'FN rate':>8s}  {'FP rate':>8s}")
print("-" * 30)
for i in range(num_classes):
    name = LABEL_NAMES[i]
    fn = final_metrics['per_class_fn'][i]
    fp = final_metrics['per_class_fp'][i]
    print(f"{name:<8s}  {fn:>8.4f}  {fp:>8.4f}")

print(f"\nMacro FN rate: {final_metrics['fn_rate']:.4f}")
print(f"Macro FP rate: {final_metrics['fp_rate']:.4f}")

## 11. Save & download the trained model

The best model (by validation F1) was already saved during training. The cell below creates a zip for easy download.

In [ ]:
import shutil

shutil.make_archive("/kaggle/working/ecg_model", "zip", "/kaggle/working/", "best_model.pth")
print("Download: /kaggle/working/ecg_model.zip")
print("\nPlace the .pth file into your local project at:  models/weights/best_model.pth")
print("Then set MOCK_INFERENCE=false in .env to use the real model.")